# LSTM Networks

## Start with the problem: what if useful evidence appeared much earlier?

A vanilla recurrent network repeatedly mixes its new input with its current hidden state. Across many steps, the effect of an early clue can shrink or become tangled with later details. Long short-term memory (LSTM) networks give the recurrence an explicit cell state and learned gates that control what to keep, write, and reveal.

An LSTM does not guarantee that it will remember anything in particular. It learns gate values from data. This lesson follows one small transition numerically, then connects the arithmetic to sequence tasks and to the limits of recurrent memory.

## Separate the memory cell from the visible hidden state

At step $t$, concatenate the current input $x_t$ and previous hidden state $h_{t-1}$, then compute four affine transforms. For a compact scalar example, write them as:

$$f_t=\sigma(a_f),\quad i_t=\sigma(a_i),\quad \tilde c_t=\tanh(a_c),\quad o_t=\sigma(a_o).$$

The forget gate $f_t$ scales the old cell, the input gate $i_t$ scales the candidate write $\tilde c_t$, and the output gate $o_t$ controls the exposed hidden state:

$$c_t=f_t c_{t-1}+i_t\tilde c_t,\qquad h_t=o_t\tanh(c_t).$$

The cell $c_t$ is the running memory channel; $h_t$ is the state passed to the next transition and commonly used for prediction. In a real layer, these are vectors and each gate has its own learned weight matrices and biases.

## Work one transition by hand

Suppose the prior cell is $c_{t-1}=0.5$ and the gate pre-activations are $a_f=0$, $a_i=0$, $a_c=1$, and $a_o=0$. Since $\sigma(0)=0.5$ and $\tanh(1)\approx0.7616$, the gates are $f_t=i_t=o_t=0.5$ and $\tilde c_t\approx0.7616$.

The new cell is $c_t=0.5(0.5)+0.5(0.7616)=0.6308$. The visible state is $h_t=0.5\tanh(0.6308)\approx0.2793$. The old memory contributes $0.25$; the candidate contributes about $0.3808$. If the forget gate were near one and the input gate near zero, the cell would mostly carry the old value forward. Gates are continuous controls, not discrete switches.

## Why the cell path can help gradients

Holding gate values fixed for intuition, the direct derivative of the new cell with respect to the old cell is $\partial c_t/\partial c_{t-1}=f_t$. Across several steps, one component of the gradient path is multiplied by the forget gates along the way. If those gates stay close to one, that path can preserve a useful signal longer than repeatedly multiplying by a saturated tanh slope.

This is a helpful route, not a promise of perfect gradients. The gates also depend on learned parameters and prior states, and gradients can still shrink, grow, or become poorly conditioned. LSTMs are trained with backpropagation through time, just like other recurrent networks.

## Compare architectures and choose a sequence readout

| Model | State update | Typical strength |
| --- | --- | --- |
| Vanilla RNN | Repeated nonlinear mixing of input and hidden state | Compact baseline for short dependencies |
| LSTM | Gated cell update plus gated hidden readout | Flexible control over retained and exposed information |
| GRU | Gated hidden-state update without a separate cell | A simpler gated alternative |

For sequence labeling, use each $h_t$ to predict a label at that position. For sequence classification, a model may use a final state, a pooled set of states, or an attention-based summary. Padding and variable lengths require masks or packed sequences so padded positions do not act like real observations.

## Where LSTMs fit

LSTMs have been used for speech, sensor streams, event histories, and language tasks where sequential processing is useful. They process one step at a time, which can make long sequences slower to parallelize than Transformer layers. They are not automatically better than a vanilla RNN, a GRU, or a Transformer; compare on the task, data size, latency, and compute budget.

**Takeaway:** the LSTM adds a cell state with learned forget, input, and output gates. The gates make information flow controllable, while learning still determines whether the model uses that control well.